In [ ]:
# Use this script after updating the demographics data in a local gdb but before uploading it to AGOL.
# You will use this script to take the demographics data and pivot it by age. Once you're done with this script
# (and any others that need to be run for this data), you can then upload both the tall and the pivoted data at once.

# Background:
# The original demographic data is formatted such that each different combination of geography, year, age group, 
# gender, and race are different records. This is fine for a table (though it makes for a very long table), but
# it's way too much data to display as a joined hosted feature layer. For this reason, on the data portal, pivoted
# data is used for the map.

# Ideas for improvement:
# Bypass Excel. I'm only using it because this works well enough and I have other stuff to do.

In [ ]:
# Used for re-shaping the table
import pandas as pd

# Used for working with Excel
import openpyxl 

In [ ]:
# You will need to have an ArcGIS Pro project open
aprx = arcpy.mp.ArcGISProject("CURRENT")

# In that project, you will need to have a map called "Map" (or change this name)
m = aprx.listMaps("Map")[0]

# In that map, you will need to have a table called "Demographics" (or change this name)
demoData = m.listTables("Demographics")[0]

In [ ]:
# Geodatabase location & where you want to save any Excel files
location = r"C:\Users\JohnsonN35\Local_work\Copied_from_local_work\MiTracking_Exploration"

# File name
file = "Demographics_pivoted_by_age"

# Excel extension
ext = ".xlsx"

# Geodatabase name (this script assumes your gdb is in the same place as where you want to save any Excel files)
gdb = "MiTracking_Population_Demographics.gdb"

In [ ]:
# Create a definition query for only the all-genders and all-races records
demoDataDQ = '"Gender" = \'All\' And "Race" = \'All\''

#Set the definition query
demoData.definitionQuery = demoDataDQ

In [ ]:
# Create a pandas dataframe

fields = ['FIPS_Table', 'Year', 'Age_Group_Num', 'Population_Estimate'] # Leaving out gender and race

df = pd.DataFrame.from_records(data = arcpy.da.SearchCursor(demoData, fields), columns = fields)

df.head()

In [ ]:
# Create a wide dataframe

df_wide = df.pivot(index = ['FIPS_Table', 'Year'], columns = ['Age_Group_Num'], values = ['Population_Estimate'])

df_wide.reset_index(inplace = True)

df_wide.head()

In [ ]:
# Export to an Excel spreadsheet

name = file + "_pandas"

df_wide.to_excel(location + "/" + name + ext, sheet_name = 'Sheet1')

In [ ]:
# Clean up the Excel file (we can't use the current table structure in a feature class)

from openpyxl import load_workbook

outpath = location + "/" + name + ext
wb = load_workbook(outpath)
ws = wb.active

# Write the values of cells B1 (2nd column) - C1 (3rd column) into cells B2 - C2
for index, row in enumerate(ws.rows, start = 2):
        if index > 3: break
        ws.cell(row = 2, column = index).value = ws.cell(row = 1, column = index).value

# Unmerge value cells (causes issues with deleting rows in next step)
ws.unmerge_cells('D1:V1')

# Delete rows and columns we don't need and that can't be imported into a feature class
ws.delete_cols(1)
ws.delete_rows(1)
ws.delete_rows(2) # Once we delete the first row, the original third row becomes the second row
wb.save(outpath)

In [ ]:
# Create a new table to hold the wide sheet

tableName = "Demographics_pivoted_by_age"

arcpy.management.CreateTable(location + "/" + gdb, tableName)

In [ ]:
# Create the fields for the table

arcpy.management.AddFields(tableName, "\
FIPS_Table TEXT 'FIPS' 7 # #;\
Year SHORT Year # # #;\
Estimate_0 LONG 'All Ages' # # #;\
Estimate_1 LONG 'Age 0-4' # # #;\
Estimate_2 LONG 'Age 5-9' # # #;\
Estimate_3 LONG 'Age 10-14' # # #;\
Estimate_4 LONG 'Age 15-19' # # #;\
Estimate_5 LONG 'Age 20-24' # # #;\
Estimate_6 LONG 'Age 25-29' # # #;\
Estimate_7 LONG 'Age 30-34' # # #;\
Estimate_8 LONG 'Age 35-39' # # #;\
Estimate_9 LONG 'Age 40-44' # # #;\
Estimate_10 LONG 'Age 45-49' # # #;\
Estimate_11 LONG 'Age 50-54' # # #;\
Estimate_12 LONG 'Age 55-59' # # #;\
Estimate_13 LONG 'Age 60-64' # # #;\
Estimate_14 LONG 'Age 65-69' # # #;\
Estimate_15 LONG 'Age 70-74' # # #;\
Estimate_16 LONG 'Age 75-79' # # #;\
Estimate_17 LONG 'Age 80-84' # # #;\
Estimate_18 LONG 'Age 85 or Older' # # #;\
")

In [ ]:
# Append wide sheet to new table

spreadsheet = location + "\\" + name + ext + "\\Sheet1$"

fieldMapping = '\
    FIPS_Table "FIPS" true true false 7 Text 0 0,First,#,spreadsheet,FIPS_Table,0,254;\
    Year "Year" true true false 2 Short 0 0,First,#,spreadsheet,Year,-1,-1;\
    Estimate_0 "All Ages" true true false 4 Long 0 0,First,#,spreadsheet,F0,-1,-1;\
    Estimate_1 "Age 0-4" true true false 4 Long 0 0,First,#,spreadsheet,F1,-1,-1;\
    Estimate_2 "Age 5-9" true true false 4 Long 0 0,First,#,spreadsheet,F2,-1,-1;\
    Estimate_3 "Age 10-14" true true false 4 Long 0 0,First,#,spreadsheet,F3,-1,-1;\
    Estimate_4 "Age 15-19" true true false 4 Long 0 0,First,#,spreadsheet,F4,-1,-1;\
    Estimate_5 "Age 20-24" true true false 4 Long 0 0,First,#,spreadsheet,F5,-1,-1;\
    Estimate_6 "Age 25-29" true true false 4 Long 0 0,First,#,spreadsheet,F6,-1,-1;\
    Estimate_7 "Age 30-34" true true false 4 Long 0 0,First,#,spreadsheet,F7,-1,-1;\
    Estimate_8 "Age 35-39" true true false 4 Long 0 0,First,#,spreadsheet,F8,-1,-1;\
    Estimate_9 "Age 40-44" true true false 4 Long 0 0,First,#,spreadsheet,F9,-1,-1;\
    Estimate_10 "Age 45-49" true true false 4 Long 0 0,First,#,spreadsheet,F10,-1,-1;\
    Estimate_11 "Age 50-54" true true false 4 Long 0 0,First,#,spreadsheet,F11,-1,-1;\
    Estimate_12 "Age 55-59" true true false 4 Long 0 0,First,#,spreadsheet,F12,-1,-1;\
    Estimate_13 "Age 60-64" true true false 4 Long 0 0,First,#,spreadsheet,F13,-1,-1;\
    Estimate_14 "Age 65-69" true true false 4 Long 0 0,First,#,spreadsheet,F14,-1,-1;\
    Estimate_15 "Age 70-74" true true false 4 Long 0 0,First,#,spreadsheet,F15,-1,-1;\
    Estimate_16 "Age 75-79" true true false 4 Long 0 0,First,#,spreadsheet,F16,-1,-1;\
    Estimate_17 "Age 80-84" true true false 4 Long 0 0,First,#,spreadsheet,F17,-1,-1;\
    Estimate_18 "Age 85 or Older" true true false 4 Long 0 0,First,#,spreadsheet,F18,-1,-1\
    '

arcpy.management.Append(spreadsheet, tableName, "NO_TEST", fieldMapping)